# Salary Prediction using Ensemble Learning

**Name:** _(Hasratbeer kaur)_
**Course:** _(gen ai)_
**Dataset:** ds_salaries.csv (Data Science Job Salaries)

### Aim
To predict salary (in USD) of data science jobs using ensemble learning models - Random Forest and Gradient Boosting.

### Steps
1. Load and look at the dataset
2. Clean the data and encode categorical columns
3. Train a simple baseline model (Linear Regression)
4. Train ensemble models (Random Forest, Gradient Boosting)
5. Compare which model works best

### How to run in Colab
- Upload this notebook to Colab
- Run the cells one by one from top to bottom
- When it asks to upload a file, select `ds_salaries.csv`

## Step 1: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style("whitegrid")
RANDOM_STATE = 42

In [ ]:
# upload the dataset (choose ds_salaries.csv when it asks)
from google.colab import files
uploaded = files.upload()

## Step 2: Load the dataset

In [ ]:
df = pd.read_csv("ds_salaries.csv")
print(df.shape)   # rows, columns
df.head()

## Step 3: A quick look at the data (EDA)

In [ ]:
df.info()

In [ ]:
print(df.isnull().sum())   # check missing values

In [ ]:
# salary distribution
plt.figure(figsize=(8,5))
sns.histplot(df["salary_in_usd"], bins=40, kde=True)
plt.title("Salary Distribution (USD)")
plt.show()

In [ ]:
# average salary by experience level
plt.figure(figsize=(7,5))
sns.barplot(data=df, x="experience_level", y="salary_in_usd", estimator=np.mean,
            order=["EN","MI","SE","EX"])
plt.title("Average Salary by Experience Level")
plt.show()

## Step 4: Preprocessing

Dropping `salary` and `salary_currency` since `salary_in_usd` already gives the normalized value we want to predict.
Converting text columns to numbers using Label Encoding.

In [ ]:
data = df.drop(columns=["salary", "salary_currency"]).copy()

categorical_cols = ["experience_level", "employment_type", "job_title",
                    "employee_residence", "company_location", "company_size"]

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])

data.head()

In [ ]:
X = data.drop(columns=["salary_in_usd"])
y = data["salary_in_usd"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

## Step 5: Baseline model (Linear Regression)

In [ ]:
results = {}

def evaluate_model(name, model):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2}
    print(name, "-> RMSE:", round(rmse), " MAE:", round(mae), " R2:", round(r2, 3))
    return model

lr = evaluate_model("Linear Regression", LinearRegression())

## Step 6: Ensemble Models

In [ ]:
rf = evaluate_model(
    "Random Forest",
    RandomForestRegressor(n_estimators=200, max_depth=10, random_state=RANDOM_STATE)
)

In [ ]:
gb = evaluate_model(
    "Gradient Boosting",
    GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=RANDOM_STATE)
)

## Step 7: Comparing the models

In [ ]:
results_df = pd.DataFrame(results).T.sort_values("R2", ascending=False)
results_df

In [ ]:
plt.figure(figsize=(7,5))
sns.barplot(x=results_df.index, y=results_df["R2"])
plt.title("Model Comparison (R2 score)")
plt.ylabel("R2 Score")
plt.show()

best_model_name = results_df["R2"].idxmax()
print("Best model:", best_model_name)

## Step 8: Feature importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(7,5))
sns.barplot(x=importances.values, y=importances.index)
plt.title("Feature Importance (Random Forest)")
plt.show()

## Conclusion

- Random Forest and Gradient Boosting (ensemble models) performed better than plain Linear Regression.
- Experience level and job title are the most important factors affecting salary.
- These ensemble models combine many decision trees, which helps them capture non-linear patterns better than a simple linear model.